# Evaluation of Matrix Pseudo-Differential Operators (`psiop.py`)

This notebook demonstrates six non-trivial examples evaluating $2 \times 2$ matrix-valued pseudo-differential operator functionalities using the `MatrixPseudoDifferentialOperator` class from `psiop`.

## 1. Imports and Setup

Import essential numerical, symbolic, and plotting libraries, and configure global visualization parameters.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import scipy.sparse as sparse
import scipy.sparse.linalg as spla

from psiop import MatrixPseudoDifferentialOperator, PseudoDifferentialOperator

# Matplotlib configuration for consistent plot rendering
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})


## 2. Helper Utilities

Define utility functions for error checking, test reporting, spectral derivatives, and matrix evaluations.

In [ ]:
def rel_l2_error(a, b):
    """Compute relative L2 error between arrays a and b."""
    a, b = np.asarray(a), np.asarray(b)
    nb = np.linalg.norm(b)
    return float(np.linalg.norm(a - b) / (nb if nb > 0 else 1.0))


def report(name, err, tol=1e-9):
    """Report test result and pass/fail status based on tolerance."""
    status = "PASS" if err < tol else "FAIL"
    print(f"  [{status}] {name:<42s} rel. L2 err = {err:.3e}")


def spectral_derivative(u, xg):
    """1D periodic spectral d/dx via FFT."""
    N, dx = len(u), xg[1] - xg[0]
    k = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
    return np.fft.ifft(1j * k * np.fft.fft(u))


def spectral_derivative_axis(u, axis, grid):
    """Periodic spectral derivative along one axis of a 2D array."""
    N, d = u.shape[axis], grid[1] - grid[0]
    k = 2.0 * np.pi * np.fft.fftfreq(N, d=d)
    shape = [1] * u.ndim
    shape[axis] = N
    return np.fft.ifft(1j * k.reshape(shape) * np.fft.fft(u, axis=axis), axis=axis)


def matrix_frobenius_field(M_expr, syms, grids):
    """Evaluate a sympy matrix of symbols on grids and compute Frobenius norm field."""
    fns = [sp.lambdify(syms, M_expr[i, j], "numpy")
           for i in range(M_expr.shape[0]) for j in range(M_expr.shape[1])]
    norm2 = np.zeros(grids[0].shape)
    for f in fns:
        val = np.broadcast_to(np.asarray(f(*grids), dtype=complex), grids[0].shape)
        norm2 += np.abs(val) ** 2
    return np.sqrt(norm2)


## Example 1: 1D Acoustic Interface

Evaluates a variable-coefficient hyperbolic system:
$$ P = \begin{pmatrix} 0 & c(x)\xi \\ c(x)\xi & 0 \end{pmatrix}, \quad c(x) = 2 + \sin(x) $$

In [ ]:
def example_1_acoustic_interface():
    print("\n" + "=" * 78)
    print("Example 1 - 1D acoustic system  P = [[0, c(x) xi], [c(x) xi, 0]], c = 2 + sin(x)")
    print("=" * 78)

    # 1. Define symbolic variables and operator
    x, xi = sp.symbols("x xi", real=True)
    c = 2 + sp.sin(x)
    P = sp.Matrix([[0, c * xi], [c * xi, 0]])
    op = MatrixPseudoDifferentialOperator(P, [x])  # KN quantization, Peetre backend

    # 2. Setup spatial domain and wavevector grids
    N, L = 512, np.pi
    xg = np.linspace(-L, L, N, endpoint=False)
    dx = xg[1] - xg[0]
    kx = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)

    # 3. Create input test fields
    u1 = np.exp(-6.0 * (xg + 1.2) ** 2)
    u2 = np.exp(-6.0 * (xg - 0.8) ** 2) * np.cos(4.0 * xg)

    # Apply operator to vector field [u1, u2]
    v1, v2 = op.apply([u1, u2], xg, kx, freq_window=None, clamp=np.inf)

    # 4. Compare with exact KN reference solution
    cv = 2.0 + np.sin(xg)
    report("(P u)_1  vs  -i c(x) u2'", rel_l2_error(v1, -1j * cv * spectral_derivative(u2, xg)))
    report("(P u)_2  vs  -i c(x) u1'", rel_l2_error(v2, -1j * cv * spectral_derivative(u1, xg)))

    # 5. Eigen-symbol hyperbolicity check
    x_sel = np.array([-2.5, -1.0, 0.0, 1.5, 3.0])
    xi_line = np.linspace(-8.0, 8.0, 400)
    eigvals, eigvecs = op.eigen_symbol(x_sel[:, None], xi_line[None, :])
    max_imag = float(np.max(np.abs(eigvals.imag)))
    print(f"  max |Im(lambda_pm)| = {max_imag:.2e}  ->  real spectrum: system hyperbolic")

    # 6. Visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(xg, cv, "b", lw=2)
    axes[0].set_xlabel("x"); axes[0].set_ylabel("c(x)")
    axes[0].set_title("Wave speed c(x) = 2 + sin(x)")

    cmap = plt.get_cmap("tab10")
    for i, x0 in enumerate(x_sel):
        col = cmap(i)
        axes[1].plot(xi_line, eigvals[i, :, 0].real, color=col, lw=1.8,
                     label=f"x = {x0:.1f}")
        axes[1].plot(xi_line, eigvals[i, :, 1].real, color=col, lw=1.8, ls="--")
    axes[1].set_xlabel(r"$\xi$"); axes[1].set_ylabel(r"$\lambda_\pm$")
    axes[1].set_title(r"Eigenvalues $\lambda_\pm(x,\xi) = \pm c(x)|\xi|$")
    axes[1].legend(fontsize=8)

    axes[2].plot(xg, np.abs(u1), "C0", lw=1.2, label=r"$|u_1|$ (input)")
    axes[2].plot(xg, np.abs(u2), "C1", lw=1.2, label=r"$|u_2|$ (input)")
    axes[2].plot(xg, np.abs(v1), "C0", lw=2.2, ls="--", label=r"$|(Pu)_1|$ (output)")
    axes[2].plot(xg, np.abs(v2), "C1", lw=2.2, ls="--", label=r"$|(Pu)_2|$ (output)")
    axes[2].set_xlabel("x"); axes[2].set_ylabel("amplitude")
    axes[2].set_title("apply() on a 2-component field")
    axes[2].legend(fontsize=8)

    fig.suptitle("Example 1 - variable-coefficient 2x2 acoustic operator", y=1.02)
    fig.tight_layout()
    plt.show()

example_1_acoustic_interface()


## Example 2: Non-Commutative Symbolic Calculus

Tests non-commutative matrix composition and asymptotic expansions for symbol commutators $[P, Q]$.

In [ ]:
def example_2_noncommutative_calculus():
    print("\n" + "=" * 78)
    print("Example 2 - matrix composition & commutator (non-commutative calculus)")
    print("=" * 78)

    x, xi = sp.symbols("x xi", real=True)

    # ---- (a) Constant-coefficient symbols: composition exact at any order ----
    A = sp.Matrix([[xi, 1], [0, xi ** 2]])
    B = sp.Matrix([[xi ** 2, 0], [xi, 1]])
    opA = MatrixPseudoDifferentialOperator(A, [x])
    opB = MatrixPseudoDifferentialOperator(B, [x])

    C_ab = opA.compose_asymptotic(opB, order=3, mode="kn")
    C_ba = opB.compose_asymptotic(opA, order=3, mode="kn")
    C_ref = sp.simplify(A * B)

    diffs = [sp.simplify(C_ab[i, j] - C_ref[i, j]) for i in range(2) for j in range(2)]
    ok_exact = all(d == 0 for d in diffs)
    noncomm = sp.simplify(C_ab - C_ba)
    ok_noncomm = any(sp.simplify(noncomm[i, j]) != 0 for i in range(2) for j in range(2))
    print(f"  compose(A, B, order=3) == A*B exactly : {ok_exact}")
    print(f"  A o B != B o A (matrix non-commutativity) : {ok_noncomm}")
    print("  A*B =")
    sp.pprint(C_ref)

    # ---- (b) Variable coefficients: commutator nonzero at order 0 ----
    Pm = sp.Matrix([[0, -sp.I * xi + x], [-sp.I * xi - x, 0]])   # Dirac-type
    Qm = sp.Matrix([[xi ** 2, 0], [0, x ** 2]])
    opP = MatrixPseudoDifferentialOperator(Pm, [x])
    opQ = MatrixPseudoDifferentialOperator(Qm, [x])

    comm0 = sp.simplify(opP.commutator_symbolic(opQ, order=0, mode="kn"))
    comm2 = sp.simplify(opP.commutator_symbolic(opQ, order=2, mode="kn"))
    print("  [P, Q] at order 0 (pure matrix commutator, already nonzero) =")
    sp.pprint(comm0)
    print("  [P, Q] at order 2 (with microlocal derivative corrections) =")
    sp.pprint(comm2)

    # ---- Plot Frobenius norm of the commutator in phase space ----
    Xg, XIg = np.meshgrid(np.linspace(-3, 3, 301), np.linspace(-6, 6, 301), indexing="ij")
    f0 = matrix_frobenius_field(comm0, (x, xi), (Xg, XIg))
    f2 = matrix_frobenius_field(comm2, (x, xi), (Xg, XIg))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
    for ax, f, t in zip(axes, (f0, f2),
                        ("order 0 (matrix commutator)", "order 2 (+ derivative terms)")):
        pcm = ax.pcolormesh(Xg, XIg, np.log10(1.0 + f), cmap="magma", shading="auto")
        fig.colorbar(pcm, ax=ax, label=r"$\log_{10}(1+\|[P,Q]\|_F)$")
        ax.set_xlabel("x"); ax.set_ylabel(r"$\xi$")
        ax.set_title(f"matrix commutator, {t}")
    
    fig.suptitle(r"Example 2 - Frobenius norm of $[P,Q](x,\xi)$,  "
                 r"$P = [[0, -i\xi+x], [-i\xi-x, 0]]$, "
                 r"$Q=\mathrm{diag}(\xi^2, x^2)$", y=1.02)
    fig.tight_layout()
    plt.show()

example_2_noncommutative_calculus()


## Example 3: Dirac Operator with Domain-Wall Mass

Analyzes the avoided crossing in eigenvalue structure for a domain-wall mass operator:
$$ P = \begin{pmatrix} m(x) & \xi \\ \xi & -m(x) \end{pmatrix}, \quad m(x) = \tanh(x) $$

In [ ]:
def example_3_dirac_domain_wall():
    print("\n" + "=" * 78)
    print("Example 3 - Dirac operator  P = [[m(x), xi], [xi, -m(x)]],  m(x) = tanh(x)")
    print("=" * 78)

    x, xi = sp.symbols("x xi", real=True)
    m = sp.tanh(x)
    Pd = sp.Matrix([[m, xi], [xi, -m]])
    op = MatrixPseudoDifferentialOperator(Pd, [x])

    # Grid setup
    N, L = 1024, 6.0
    xg = np.linspace(-L, L, N, endpoint=False)
    dx = xg[1] - xg[0]
    kx = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)

    # Input spinor components
    k0 = 6.0
    u1 = np.exp(-(xg + 1.5) ** 2) * np.exp(1j * k0 * xg)
    u2 = 0.6 * np.exp(-(xg - 1.0) ** 2 / 0.5)

    v1, v2 = op.apply([u1, u2], xg, kx, freq_window=None, clamp=np.inf)

    # Exact reference comparison
    mv = np.tanh(xg)
    report("(P u)_1  vs  m u1 - i u2'", rel_l2_error(v1, mv * u1 - 1j * spectral_derivative(u2, xg)))
    report("(P u)_2  vs  -i u1' - m u2", rel_l2_error(v2, -1j * spectral_derivative(u1, xg) - mv * u2))

    # Evaluate spectral gap
    Xg, XIg = np.meshgrid(np.linspace(-4, 4, 300), np.linspace(-6, 6, 250), indexing="ij")
    eigvals, _ = op.eigen_symbol(Xg, XIg)
    lam_plus = eigvals[..., 0].real
    lam_minus = eigvals[..., 1].real
    gap = float(np.min(lam_plus - lam_minus))
    print(f"  minimal spectral gap on the plotted window: {gap:.3e} "
          f"(gap closes at x=0, xi=0 since m(0)=0)")

    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    vmax = max(np.abs(lam_minus).max(), np.abs(lam_plus).max())
    pcm0 = axes[0, 0].pcolormesh(Xg, XIg, lam_minus, cmap="RdBu_r",
                                 vmin=-vmax, vmax=vmax, shading="auto")
    fig.colorbar(pcm0, ax=axes[0, 0], label=r"$\lambda_-$")
    axes[0, 0].plot([0], [0], "kx", ms=10, mew=2)
    axes[0, 0].set_title(r"$\lambda_-(x,\xi)=-\sqrt{\xi^2+m(x)^2}$")

    pcm1 = axes[0, 1].pcolormesh(Xg, XIg, lam_plus, cmap="RdBu_r",
                                 vmin=-vmax, vmax=vmax, shading="auto")
    fig.colorbar(pcm1, ax=axes[0, 1], label=r"$\lambda_+$")
    axes[0, 1].plot([0], [0], "kx", ms=10, mew=2)
    axes[0, 1].set_title(r"$\lambda_+(x,\xi)=+\sqrt{\xi^2+m(x)^2}$")
    for ax in axes[0]:
        ax.set_xlabel("x"); ax.set_ylabel(r"$\xi$")

    axes[1, 0].plot(xg, np.abs(u1), "C0", label=r"$|u_1|$")
    axes[1, 0].plot(xg, np.abs(u2), "C1", label=r"$|u_2|$")
    axes[1, 0].plot(xg, mv, "k--", lw=1, label="m(x)")
    axes[1, 0].set_title("input spinor + mass profile")

    axes[1, 1].plot(xg, np.abs(v1), "C0", label=r"$|(Pu)_1|$")
    axes[1, 1].plot(xg, np.abs(v2), "C1", label=r"$|(Pu)_2|$")
    axes[1, 1].set_title("output spinor after apply()")
    for ax in axes[1]:
        ax.set_xlabel("x"); ax.legend(fontsize=8)

    fig.suptitle("Example 3 - Dirac domain wall: avoided crossing & spinor action", y=1.0)
    fig.tight_layout()
    plt.show()

example_3_dirac_domain_wall()


## Example 4: 2D Massless Dirac Symbol

Evaluates the 2D massless Dirac operator, confirming that $P \circ P = (\xi^2 + \eta^2)I = -\Delta$.

In [ ]:
x, y = sp.symbols("x y", real=True)
xi, eta = sp.symbols("xi eta", real=True)
P2 = sp.Matrix([[xi, eta], [eta, -xi]])
op = MatrixPseudoDifferentialOperator(P2, [x, y])

# Constant-coefficient: Weyl == KN == matrix product
comp_weyl = op.compose_asymptotic(op, order=2, mode='weyl')
comp_kn   = op.compose_asymptotic(op, order=2, mode='kn')
target    = (xi**2 + eta**2) * sp.eye(2)

print("Weyl P∘P == (ξ²+η²)I:", sp.simplify(comp_weyl - target) == sp.zeros(2))
print("KN   P∘P == (ξ²+η²)I:", sp.simplify(comp_kn - target) == sp.zeros(2))

# Variable-coefficient: Weyl ≠ KN (correction terms appear)
V = sp.Matrix([[sp.sin(x), 0], [0, sp.cos(y)]])
opV = MatrixPseudoDifferentialOperator(V, [x, y])
comp_weyl_v = sp.simplify(op.compose_asymptotic(opV, order=1, mode='weyl'))
comp_kn_v   = sp.simplify(op.compose_asymptotic(opV, order=1, mode='kn'))
print("P∘V (Weyl, order 1):")
sp.pprint(comp_weyl_v)
print("P∘V (KN, order 1):")
sp.pprint(comp_kn_v)

In [ ]:
def example_4_2d_dirac_cone():
    print("\n" + "=" * 78)
    print("Example 4 - 2D massless Dirac  P = [[xi, eta], [eta, -xi]]")
    print("=" * 78)

    x, y = sp.symbols("x y", real=True)
    xi, eta = sp.symbols("xi eta", real=True)
    P2 = sp.Matrix([[xi, eta], [eta, -xi]])
    op = MatrixPseudoDifferentialOperator(P2, [x, y])

    N, L = 128, np.pi
    xg = np.linspace(-L, L, N, endpoint=False)
    yg = np.linspace(-L, L, N, endpoint=False)
    dx, dy = xg[1] - xg[0], yg[1] - yg[0]
    kx = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
    ky = 2.0 * np.pi * np.fft.fftfreq(N, d=dy)

    X, Y = np.meshgrid(xg, yg, indexing="ij")
    u1 = np.exp(-2.0 * (X ** 2 + Y ** 2))
    u2 = np.exp(-2.0 * ((X - 0.7) ** 2 + (Y + 0.5) ** 2))

    v1, v2 = op.apply([u1, u2], xg, kx, y_grid=yg, ky=ky,
                      freq_window=None, clamp=np.inf)

    # Reference check
    du1x, du1y = spectral_derivative_axis(u1, 0, xg), spectral_derivative_axis(u1, 1, yg)
    du2x, du2y = spectral_derivative_axis(u2, 0, xg), spectral_derivative_axis(u2, 1, yg)
    report("(P u)_1  vs  -i (dx u1 + dy u2)", rel_l2_error(v1, -1j * (du1x + du2y)))
    report("(P u)_2  vs  -i dy u1 + i dx u2", rel_l2_error(v2, -1j * du1y + 1j * du2x))

    # Symbolic check: P o P = (xi^2 + eta^2) I
    comp = op.compose_asymptotic(op, order=2, mode="kn")
    target = (xi ** 2 + eta ** 2) * sp.eye(2)
    ok = all(sp.simplify(comp[i, j] - target[i, j]) == 0 for i in range(2) for j in range(2))
    print(f"  P o P == (xi^2 + eta^2) I symbolically (mode='kn') : {ok}")

    # Numerical check: apply twice == -Laplacian
    w1, w2 = op.apply([v1, v2], xg, kx, y_grid=yg, ky=ky,
                      freq_window=None, clamp=np.inf)
    KXf, KYf = np.meshgrid(kx, ky, indexing="ij")
    minus_lap = lambda u: -np.fft.ifft2((KXf ** 2 + KYf ** 2) * np.fft.fft2(u))
    report("P(P u1)  vs  -Laplacian u1", rel_l2_error(w1, minus_lap(u1)))
    report("P(P u2)  vs  -Laplacian u2", rel_l2_error(w2, minus_lap(u2)))

    # Variable-coefficient composition
    V = sp.Matrix([[sp.sin(x), 0], [0, sp.cos(y)]])
    opV = MatrixPseudoDifferentialOperator(V, [x, y])
    comp_v = sp.simplify(op.compose_asymptotic(opV, order=1, mode="kn"))
    print("  P o V at order 1 =")
    sp.pprint(comp_v)

    op.compose_asymptotic(op, order=1, mode="weyl")

    # Figure 1: Dirac cone surface plot
    KXc, KYc = np.meshgrid(np.linspace(-8, 8, 120), np.linspace(-8, 8, 120), indexing="ij")
    eigvals, _ = op.eigen_symbol(0.0, 0.0, KXc, KYc)

    fig = plt.figure(figsize=(12, 5))
    for k, (lam, t, sgn) in enumerate((
            (eigvals[..., 0].real, r"$\lambda_+$", "+"),
            (eigvals[..., 1].real, r"$\lambda_-$", "-"))):
        ax = fig.add_subplot(1, 2, k + 1, projection="3d")
        ax.plot_surface(KXc, KYc, lam, cmap="viridis", alpha=0.9)
        ax.set_xlabel(r"$\xi$"); ax.set_ylabel(r"$\eta$"); ax.set_zlabel(t)
        ax.set_title(rf"{t} = {sgn}\sqrt{{\xi^2+\eta^2}}")
    fig.suptitle("Example 4 - Dirac cone of the 2x2 symbol (eigen_symbol)", y=1.0)
    fig.tight_layout()
    plt.show()

    # Figure 2: Input / output scalar fields
    fig2, axes2 = plt.subplots(2, 2, figsize=(10, 8), sharex=True, sharey=True)
    for ax, F, t in zip(axes2.ravel(), (u1, u2, v1, v2),
                        (r"$|u_1|$", r"$|u_2|$", r"$|(Pu)_1|$", r"$|(Pu)_2|$")):
        pcm = ax.pcolormesh(X, Y, np.abs(F), cmap="inferno", shading="auto")
        fig2.colorbar(pcm, ax=ax)
        ax.set_title(t); ax.set_xlabel("x"); ax.set_ylabel("y")
    fig2.suptitle("Example 4 - apply() of the 2D Dirac symbol on a vector field", y=1.0)
    fig2.tight_layout()
    plt.show()

example_4_2d_dirac_cone()
